In [7]:
import requests
import pandas as pd
import re
import urllib
from tqdm import tqdm

# This notebook converts voting_id's to dataframes of votes for each period

In [8]:
with urllib.request.urlopen("https://raw.githubusercontent.com/Somon8/social_graphs_25/main/final-project/folketinget_votings_enriched.csv") as response:
    df_votings = pd.read_csv(response)

In [9]:
n_votes = len(df_votings['id_afstemning'].unique())
n_periods = df_votings['Period'].nunique()
print(f"There are {n_votes} unique votes in the dataset, across {n_periods} periods.")

There are 10304 unique votes in the dataset, across 7 periods.


In [10]:
# base_url = "https://oda.ft.dk/api/"
    
# n_votes = 500
# def get_voting_sessions_in_period():
#     # Get all voting ID´s in a specific period to pass to "get_voting_session_with_votes"
#     # voting_ids= [10375, 10376, 10377, 10378] #
#     voting_ids = range(10379-n_votes, 10379)
#     return voting_ids

def get_voting_sessions_in_period(period):
    votings_ids = df_votings[df_votings['Period'] == period]['id_afstemning'].unique() #Do i need to convert to list?
    print(f"Found {len(votings_ids)} votes in period {period}")
    return votings_ids

# BASE_URL = "https://oda.ft.dk/api/"
request_session = requests.Session()

def get_voting_session_with_votes(voting_id, session = request_session):
    base_url = "https://oda.ft.dk/api/"
    """Retrieve a complete voting session with all individual votes"""
    params = {
        '$expand': 'Stemme/Aktør',
        '$filter': f'id eq {voting_id}'
    }
    url = f"{base_url}Afstemning"
    response = session.get(url, params=params) #Use session to reuse connections and make everything run faster

    if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
        print(f"HTTP error for {voting_id}: ", response.status_code)
        print("Response text:", response.text)
        return None
    else:
        try:
            data = response.json()
        except ValueError:
            print("Error: Response is not valid JSON")
            print("Response text:", response.text)
            return None
        
        return data
    


def get_data_from_voting_session(voting_id, data_from_voting_session):
    if data_from_voting_session['value']:

        vote_data = []

        # for aktør 
        data = data_from_voting_session['value'][0].get('Stemme')
        
        for individual_vote in data:
            # return_data = individual_vote
            # return_data = individual_vote.get('Aktør').get('biografi') #This is useful for finding the information about the person who voted.

            politician_bio = individual_vote.get('Aktør').get('biografi')
            politician_party = re.search('<party>([^<]+)</party>', politician_bio).group(1) #Extract the party
            politician_name = individual_vote.get('Aktør').get('navn') #Find the name of who votes. We will use this for naming the nodes.
            vote_type = individual_vote.get('typeid') #What did they vote?
            # vote_data.append((politician_name, voting_id, {'vote_type':vote_type})) #Need to pass the last part as dictionary for networkx to understand it

            # POTENTIALLY SKIP IF THEY WERE ABSENT FOR THE VOTE
            # if vote_type == 3: #I would rather gather all the data now and then remove things later.
            #     continue
            vote_data.append((voting_id, politician_party, politician_name, vote_type)) #Will pass to DF instead, so no need for dict 
            # print(vote_data)
    
    # return vote_data
    df = pd.DataFrame(vote_data, columns = ['voting_id', 'party', 'politician', 'vote_type'])
    return df
    # return return_data #This is if we want to see example output to understand the structure.

# voting_id = 10377
# data = get_voting_session_with_votes(voting_id)
# df = get_data_from_voting_session(voting_id=voting_id, data_from_voting_session=data)
# df
# # df.head()

In [ ]:
# voting_period = 69
#Save it as a period
def get_and_save_df_from(voting_period):
    voting_sessions = get_voting_sessions_in_period(voting_period)
    df = pd.DataFrame()
    for voting_id in tqdm(voting_sessions):
        data = get_voting_session_with_votes(voting_id)
        df_addition = get_data_from_voting_session(voting_id=voting_id, data_from_voting_session=data)    
        df = pd.concat([df, df_addition])

    df.to_csv(f"./voting-data/df_votes_p{voting_period}.csv", index = False)

for period in df_votings['Period'].unique():
    # print(f'Now working on period {period}')
    get_and_save_df_from(period)


Found 1727 votes in period 68


100%|██████████| 1727/1727 [04:21<00:00,  6.61it/s]


Found 1778 votes in period 67


100%|██████████| 1778/1778 [04:52<00:00,  6.08it/s]


Found 2019 votes in period 69


...

100%|██████████| 294/294 [00:38<00:00,  7.67it/s]


Found 1838 votes in period 70


100%|██████████| 1838/1838 [05:19<00:00,  5.76it/s]


Found 1379 votes in period 71


100%|██████████| 1379/1379 [03:47<00:00,  6.07it/s]


In [12]:
df = pd.DataFrame()
for period in df_votings['Period'].unique():
    with urllib.request.urlopen(f"https://raw.githubusercontent.com/Somon8/social_graphs_25/main/final-project/voting-data/df_votes_p{period}.csv") as response:
        df_votes_from_period = pd.read_csv(response)
        df = pd.concat([df, df_votes_from_period])

df.to_csv(f"./voting-data/df_votes_all_periods.csv", index = False)